# Eksplorasi sistem fuzzy diskon barbershop

Notebook ini saya pakai untuk debug awal sebelum nulis paper:

1. Cek bentuk MF di tiga variabel
2. Sweep beberapa profil pelanggan dan lihat outputnya
3. Bandingkan fuzzy vs rule kaku di seluruh dataset

Plot final yang saya pakai di paper saya generate via `src/plots.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.fuzzy import recommend_discount, freq_curves, spend_curves, disc_curves
from src.baseline import traditional_discount
from src.data import load

## 1. Bentuk membership functions

In [ ]:
for name, fn in [('Frekuensi', freq_curves), ('Spending', spend_curves), ('Diskon', disc_curves)]:
    x, c = fn()
    plt.figure(figsize=(7, 2.5))
    for term, mf in c.items():
        plt.plot(x, mf, label=term)
    plt.title(name); plt.legend(); plt.grid(alpha=0.3)
    plt.show()

## 2. Sweep beberapa skenario

In [ ]:
skenario = [
    ('Pekerja baru, sekali cukur basic', 1, 90),
    ('Reguler 2x/bulan, masing-masing 90k', 2, 180),
    ('Eksekutif, paket 350k 1x', 1, 350),
    ('Profesional muda mingguan, total 280k', 4, 280),
    ('VIP loyal heavy', 6, 480),
]
rows = []
for name, f, s in skenario:
    rows.append({
        'Profil': name, 'freq': f, 'spend_k': s,
        'fuzzy %': recommend_discount(f, s),
        'rule kaku %': traditional_discount(f, s),
    })
pd.DataFrame(rows)

## 3. Sweep di sepanjang spending (cliff effect)

In [ ]:
spend_grid = np.linspace(0, 500, 200)
fz = [recommend_discount(2, s) for s in spend_grid]
bs = [traditional_discount(2, s) for s in spend_grid]
plt.figure(figsize=(8, 3.5))
plt.plot(spend_grid, fz, label='fuzzy')
plt.plot(spend_grid, bs, label='rule kaku', linestyle='--')
plt.xlabel('spending (ribu)'); plt.ylabel('diskon (%)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4. Statistik di dataset 100 pelanggan

In [ ]:
df = load().copy()
df['fuzzy'] = [recommend_discount(f, s) for f, s in zip(df['frequency'], df['spending_kIDR'])]
df['baseline'] = [traditional_discount(f, s) for f, s in zip(df['frequency'], df['spending_kIDR'])]
df.describe()[['frequency', 'spending_kIDR', 'fuzzy', 'baseline']]